# Rekurentne neuronske mreže

U prethodnom modulu obradili smo bogate semantičke reprezentacije teksta. Arhitektura koju smo koristili hvata agregirano značenje riječi u rečenici, ali ne uzima u obzir **redoslijed** riječi, jer operacija agregacije koja slijedi iza ugradnji uklanja te informacije iz izvornog teksta. Budući da ovi modeli ne mogu predstaviti redoslijed riječi, oni ne mogu riješiti složenije ili dvosmislene zadatke poput generiranja teksta ili odgovaranja na pitanja.

Da bismo uhvatili značenje tekstovne sekvence, koristit ćemo arhitekturu neuronske mreže nazvanu **rekurentna neuronska mreža**, ili RNN. Kada koristimo RNN, prosljeđujemo našu rečenicu kroz mrežu jedan token u isto vrijeme, i mreža proizvodi neku **stanje**, koje zatim prosljeđujemo mreži zajedno s idućim tokenom.

![Slika koja prikazuje primjer generiranja rekurentne neuronske mreže.](../../../../../translated_images/hr/rnn.27f5c29c53d727b5.webp)

S obzirom na ulaznu sekvencu tokena $X_0,\dots,X_n$, RNN kreira sekvencu blokova neuronske mreže i trenira tu sekvencu end-to-end koristeći povratno širenje greške (backpropagation). Svaki mrežni blok kao ulaz prima par $(X_i,S_i)$, i proizvodi $S_{i+1}$ kao rezultat. Završno stanje $S_n$ ili izlaz $Y_n$ ide u linearni klasifikator za proizvodnju rezultata. Svi blokovi mreže dijele iste težine, te se treniraju end-to-end koristeći jedan prolaz povratnog širenja.

> Gornja slika prikazuje rekurentnu neuronsku mrežu u odmotanom obliku (lijevo), i u kompaktnijem rekurentnom prikazu (desno). Važno je shvatiti da sve RNN ćelije imaju iste **dijeljene težine**.

Budući da se vektori stanja $S_0,\dots,S_n$ prenose kroz mrežu, RNN može naučiti sekvencijalne ovisnosti između riječi. Na primjer, kada se riječ *not* pojavi negdje u sekvenci, model može naučiti negirati određene elemente unutar vektora stanja.

Unutar svake RNN ćelije nalaze se dvije matrice težina: $W_H$ i $W_I$, te bias $b$. Na svakom RNN koraku, za dani ulaz $X_i$ i ulazno stanje $S_i$, izlazno stanje računa se kao $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, gdje je $f$ aktivacijska funkcija (često $\tanh$).

> Za probleme poput generiranja teksta (koje ćemo obraditi u sljedećoj jedinici) ili strojno prevođenje također želimo dobiti vrijednost izlaza na svakom RNN koraku. U tom slučaju, postoji i druga matrica $W_O$, a izlaz se računa kao $Y_i=f(W_O\times S_i+b_O)$.

Pogledajmo kako nam rekurentne neuronske mreže mogu pomoći u klasifikaciji našeg skupa podataka s vijestima.

> Za sandbox okruženje, potrebno je pokrenuti sljedeću ćeliju kako bismo osigurali da je potrebna biblioteka instalirana i da su podaci unaprijed učitani. Ako radite lokalno, možete preskočiti sljedeću ćeliju.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Prilikom treniranja velikih modela, dodjeljivanje memorije GPU-a može postati problem. Također ćemo možda trebati eksperimentirati s različitim veličinama minibatcha, tako da podaci stanu u memoriju našeg GPU-a, a da treniranje bude dovoljno brzo. Ako ovaj kod pokrećete na vlastitom GPU računalu, možete eksperimentirati s prilagodbom veličine minibatcha kako biste ubrzali treniranje.

> **Napomena**: Poznato je da određene verzije NVidia upravljačkih programa ne otpuštaju memoriju nakon treniranja modela. U ovom bilježniku pokrećemo nekoliko primjera, što može dovesti do toga da memorija ponestane u određenim postavkama, osobito ako radite vlastite eksperimente u sklopu istog bilježnika. Ako naiđete na neke čudne pogreške prilikom pokretanja treniranja modela, preporučujemo da ponovno pokrenete kernel bilježnika.


In [3]:
batch_size = 16
embed_size = 64

## Jednostavni RNN klasifikator

U slučaju jednostavnog RNN-a, svaka rekurentna jedinica je jednostavna linearna mreža koja prima ulazni vektor i vektor stanja te proizvodi novi vektor stanja. U Kerasu, to se može predstaviti slojem `SimpleRNN`.

Iako možemo proslijediti one-hot kodirane tokene izravno u RNN sloj, to nije dobra ideja zbog njihove visoke dimenzionalnosti. Stoga ćemo koristiti ugrađeni sloj za smanjenje dimenzionalnosti vektora riječi, potom RNN sloj, i na kraju `Dense` klasifikator.

> **Napomena**: U slučajevima kada dimenzionalnost nije tako visoka, na primjer prilikom upotrebe tokenizacije na razini znakova, može imati smisla proslijediti one-hot kodirane tokene izravno u RNN ćeliju.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Napomena:** Ovdje koristimo netreniranu embedding sloj radi jednostavnosti, ali za bolje rezultate možemo koristiti unaprijed trenirani embedding sloj koristeći Word2Vec, kao što je opisano u prethodnoj jedinici. Bilo bi dobro vježbati prilagodbu ovog koda za rad s unaprijed treniranim embeddingima.

Sada trenirajmo naš RNN. RNN-ovi su generalno prilično teški za treniranje, jer nakon što se RNN ćelije razviju po duljini sekvence, broj slojeva uključenih u backpropagaciju postaje prilično velik. Stoga moramo odabrati manju stopu učenja i trenirati mrežu na većem skupu podataka da bismo postigli dobre rezultate. To može potrajati prilično dugo, pa je korištenje GPU-a poželjno.

Kako bismo ubrzali stvari, trenirat ćemo RNN model samo na naslovima vijesti, izostavljajući opis. Možete pokušati trenirati i s opisom i vidjeti možete li dobiti da se model trenira.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Napomena** da je točnost vjerojatno niža ovdje, jer treniramo samo na naslovima vijesti.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Sada kada koristimo maskiranje, možemo trenirati model na cijelom skupu podataka naslova i opisa.

> **Napomena**: Jeste li primijetili da smo koristili vektorizator treniran na naslovima vijesti, a ne na cijelom tijelu članka? To potencijalno može uzrokovati da neki tokeni budu zanemareni, pa je bolje ponovno trenirati vektorizator. Međutim, to bi moglo imati vrlo mali učinak, pa ćemo radi jednostavnosti koristiti prethodno unaprijed trenirani vektorizator.


## LSTM: memorija dugog kratkoročnog pamćenja

Jedan od glavnih problema RNN-ova su **nestajući gradijenti**. RNN-ovi mogu biti prilično dugi i mogu imati poteškoća u propagiranju gradijenata sve do prvog sloja mreže tijekom povratnog širenja greske. Kada se to dogodi, mreža ne može naučiti odnose između udaljenih tokena. Jedan od načina da se izbjegne ovaj problem je uvođenje **eksplicitnog upravljanja stanjima** korištenjem **vrata**. Dvije najčešće arhitekture koje uvode vrata su **memorija dugog kratkoročnog pamćenja** (LSTM) i **gated relay unit** (GRU). Ovdje ćemo pokriti LSTM.

![Slika koja prikazuje primjer ćelije memorije dugog kratkoročnog pamćenja](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM mreža je organizirana na sličan način kao RNN, ali postoje dvije stanja koja se prenose s sloja na sloj: stvarno stanje $c$ i skriveni vektor $h$. U svakoj jedinici, skriveni vektor $h_{t-1}$ kombinira se s ulazom $x_t$, i zajedno kontroliraju što se događa sa stanjem $c_t$ i izlazom $h_{t}$ kroz **vrata**. Svaka vrata imaju sigmoidnu aktivaciju (izlaz u rasponu $[0,1]$), što se može smatrati kao maska po bitu kada se pomnoži sa statusnim vektorom. LSTM-ovi imaju sljedeća vrata (slijeva nadesno na gornjoj slici):
* **vrata zaborava** koja određuju koje komponente vektora $c_{t-1}$ trebamo zaboraviti, a koje proslijediti.
* **ulazna vrata** koja određuju koliko informacija iz ulaznog vektora i prethodnog skrivenog vektora treba inkorporirati u stanje.
* **izlazna vrata** koja uzimaju novi vektor stanja i odlučuju koje njegove komponente će se koristiti za proizvodnju novog skrivenog vektora $h_t$.

Komponente stanja $c$ mogu se smatrati kao zastavice koje se mogu uključiti ili isključiti. Na primjer, kad naiđemo na ime *Alice* u sekvenci, pretpostavljamo da se odnosi na ženu i podižemo zastavicu u stanju koja kaže da imamo ženski imenicu u rečenici. Kada nadalje naiđemo na riječi *and Tom*, podignut ćemo zastavicu koja kaže da imamo množinu imenica. Tako manipulirajući stanjem možemo pratiti gramatička svojstva rečenice.

> **Napomena**: Evo izvrsnog izvora za razumijevanje unutarnjih mehanizama LSTM-a: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) autora Christopher Olah.

Iako unutarnja struktura LSTM ćelije može izgledati složeno, Keras skriva tu implementaciju unutar sloja `LSTM`, tako da jedina stvar koju trebamo učiniti u gornjem primjeru jest zamijeniti rekurentni sloj:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Napomena** da je treniranje LSTM-ova također prilično sporo, i u početku treniranja možda nećete vidjeti velik porast točnosti. Možda ćete morati nastaviti trenirati neko vrijeme da biste postigli dobru točnost.


## Dvosmjerne i višeslojne RNN-ove

U dosadašnjim primjerima, rekurentne mreže rade od početka do kraja sekvence. To nam se čini prirodnim jer slijedi isti smjer u kojem čitamo ili slušamo govor. Međutim, za scenarije koji zahtijevaju slučajni pristup ulaznoj sekvenci, smislenije je izvoditi rekurentne proračune u oba smjera. RNN-ovi koji dopuštaju proračune u oba smjera nazivaju se **dvosmjernim** RNN-ovima, a mogu se kreirati omotavanjem rekurentnog sloja sa specijalnim slojem `Bidirectonal`.

> **Napomena**: Sloj `Bidirectional` pravi dvije kopije sloja unutar sebe, i postavlja `go_backwards` svojstvo jednoj od tih kopija na `True`, što ju tjera da ide u suprotnom smjeru duž sekvence.

Rekurentne mreže, jednsmjerne ili dvosmjerne, hvataju obrasce unutar sekvence i pohranjuju ih u vektore stanja ili ih vraćaju kao izlaz. Kao i kod konvolucijskih mreža, možemo izgraditi još jedan rekurentni sloj nakon prvog da bismo uhvatili višerazinske obrasce, građene od nižerazinskih obrazaca koje je prije izvukao prvi sloj. To nas dovodi do pojma **višeslojni RNN**, koji se sastoji od dva ili više rekurentnih mreža, gdje se izlaz prethodnog sloja prosljeđuje sljedećem sloju kao ulaz.

![Slika koja prikazuje višeslojni long-short-term-memory RNN](../../../../../translated_images/hr/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Slika iz [ovog sjajnog posta](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) autora Fernanda Lópeza.*

Keras olakšava konstruiranje ovih mreža jer samo trebate dodati više rekurentnih slojeva u model. Za sve slojeve osim zadnjeg, moramo specificirati parametar `return_sequences=True`, jer nam je potrebno da sloj vraća sva međustanja, a ne samo konačno stanje rekurentnog proračuna.

Napravimo dvoslojni dvosmjerni LSTM za naš problem klasifikacije.

> **Napomena** ovaj kod ponovno traje prilično dugo, ali nam daje najveću točnost do sada. Dakle, možda vrijedi pričekati i vidjeti rezultat.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN-ovi za druge zadatke

Do sada smo se fokusirali na korištenje RNN-ova za klasificiranje nizova teksta. Ali oni mogu obraditi i mnoge druge zadatke, poput generiranja teksta i strojno prevođenje &mdash; te ćemo zadatke razmotriti u sljedećoj jedinici.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
